# Queimar Legenda MULTICOR (classificação gramatical por palavra)

> 🖥️ **CPU basta** — este notebook não usa GPU. Deixe o acelerador em *Nenhum*: não muda nada aqui e poupa sua cota de GPU, que é limitada.

Pega um arquivo `.ass` (o que saiu do `caption-multicolor-generate.ipynb` ou do
`caption-multicolor-zh-generate.ipynb` — pode ser o original ou uma versão que
você corrigiu manualmente) e queima no vídeo/imagem de fundo do vídeo.

**Serve as duas variantes de idioma.** Não existe um
`caption-multicolor-zh-burn.ipynb` porque não precisa: o `.ass` de 6 idiomas
sai com `_zh` no nome, e a célula 5 lê esse sufixo pra gravar o resultado em
`<nome>_final_multicolor_zh.mp4` em vez de por cima do de 5 idiomas.

No final, duas ações **separadas**: baixar o vídeo final pra conferir, e
(só depois de confirmar que ficou bom) salvar no Drive.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1. SETUP                                                        ║
# ╚══════════════════════════════════════════════════════════════════╗
# Garante a fonte do coreano (Noto Sans CJK) instalada — sem isso, o
# libass (o que desenha a legenda em cima do vídeo) não acha os
# caracteres Hangul e mostra quadradinho (□) no lugar. Não dá pra supor
# que já vem instalada no ambiente do Colab.
!apt-get install -y -qq fonts-noto-cjk > /dev/null 2>&1

import shutil, sys
from pathlib import Path
from google.colab import drive

try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

PASTA_DRIVE_RAIZ_MODULOS = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ_MODULOS}/pipeline/modulos")
DESTINO = Path("/content/pipeline")
if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} módulos copiados")

    # ── A cópia trouxe TODOS os módulos? ───────────────────────────────────────
    # "N módulos copiados" sozinho não quer dizer nada. E o modo de falhar aqui é
    # traiçoeiro: o Drive montado do Colab popula a listagem da pasta com atraso,
    # então um copytree logo depois do mount às vezes enxerga só parte dos
    # arquivos. Já aconteceu de copiar 13 de 31 -- com visto verde -- e o notebook
    # quebrar muito depois, num import, longe da causa.
    #
    # A conferência é de três pontas, porque a causa muda o conserto:
    #   manifesto  o que o repositório tem  (versionado; chega pela cópia)
    #   Drive      o que chegou lá
    #   VM         o que a cópia desta célula trouxe
    # A conferência tem duas perguntas, e SÓ UMA delas precisa do manifesto:
    #
    #   Drive → VM   a cópia acima trouxe tudo?      dá pra ver aqui mesmo
    #   repo → Drive o Drive está em dia?            só o manifesto sabe
    #
    # A versão anterior amarrava as duas ao manifesto: sem ele, imprimia um
    # aviso e seguia SEM CONFERIR NADA. Foi assim que "✅ 13 modules copied"
    # passou com visto verde num Drive que tinha 31 -- justamente no dia em
    # que o manifesto ainda não existia. Comparar 13 com 31 nunca dependeu de
    # manifesto nenhum.
    _no_drive = {f.name for f in PASTA_MODULOS.glob("*.py")}
    _na_vm    = {f.name for f in DESTINO.glob("*.py")}

    # ── Drive → VM ────────────────────────────────────────────────────────
    # O Drive montado do Colab popula a listagem da pasta com atraso, então um
    # copytree logo depois do mount às vezes enxerga só parte dos arquivos.
    # Uma segunda passada, com o mount já quente, costuma resolver.
    _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"   ⏳ {len(_nao_copiados)} módulo(s) não vieram na 1ª passada — copiando de novo")
        for _n in _nao_copiados:
            shutil.copyfile(PASTA_MODULOS / _n, DESTINO / _n)
        _na_vm = {f.name for f in DESTINO.glob("*.py")}
        _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"\n🚨 {len(_nao_copiados)} módulo(s) estão no Drive mas não copiaram:")
        for _n in _nao_copiados:
            print(f"     {_n}")
        raise SystemExit("Rode ESTA célula de novo — o Drive montado ainda estava acordando.")
    print(f"   ✅ os {len(_no_drive)} módulos do Drive chegaram na VM")

    # ── repositório → Drive ───────────────────────────────────────────────
    _manifesto = PASTA_MODULOS / "_manifesto.txt"
    if not _manifesto.exists():
        print("   ⚠️  sem _manifesto.txt: não dá pra saber se o DRIVE está atrás")
        print("      do repositório. Ele é versionado — rode o repositorio-sincronizar.")
    else:
        _esperados = {l.strip() for l in _manifesto.read_text().splitlines()
                      if l.strip() and not l.startswith("#")}
        _fora_do_drive = sorted(_esperados - _no_drive)
        if _fora_do_drive:
            print(f"\n🚨 {len(_fora_do_drive)} módulo(s) não estão no DRIVE:")
            for _n in _fora_do_drive:
                print(f"     {_n}")
            raise SystemExit("Rode o repositorio-sincronizar.ipynb — o Drive está atrás do repositório.")
        print(f"   ✅ e batem com os {len(_esperados)} do manifesto")

    # ── O Python está segurando a versão anterior? ────────────────────────
    # Copiar arquivo novo por cima não desfaz um import já feito: o Python
    # guarda o módulo em sys.modules e reaproveita. Numa sessão longa, isso
    # faz o notebook rodar com o config.py de ontem mesmo depois de um sync
    # perfeito -- e o sintoma aparece longe da causa (nome de arquivo que
    # mudou, padrão que era pra ter mudado e não mudou). Descarregar aqui
    # equivale a reiniciar o runtime, sem perder o resto da sessão.
    _recarregar = [_n for _n, _m in list(sys.modules.items())
                   if getattr(_m, "__file__", None) and str(DESTINO) in str(_m.__file__)]
    for _n in _recarregar:
        del sys.modules[_n]
    if _recarregar:
        print(f"   ♻️  {len(_recarregar)} módulo(s) já importados foram descarregados —")
        print(f"      o import vai reler a cópia nova (rode as células seguintes de novo)")
else:
    print(f"❌ Pasta de módulos não encontrada: {PASTA_MODULOS}")
if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

# ── O ambiente combina com o que este notebook faz? ────────────────────────
# Cota de GPU do Colab é limitada e some sem aviso -- e parte da nossa foi
# gasta em notebook que não usa GPU pra nada, rodando com GPU só porque a
# seleção ficou de antes. Silencioso quando combina.
try:
    from ambiente import avisar_gpu
    avisar_gpu(precisa=False)
except Exception:
    pass

print("✅ Setup concluído")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2. CONFIGURAÇÃO                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝
NOME_ORACAO = "40_Matt_02"
PASTA_DRIVE_RAIZ = "narrated_video"

from config import PipelineConfig
from drive_utils import DriveClient


# Fundo: "imagem", "video", ou None pra detectar sozinho pelo arquivo que
# existe no Drive. Só preencha à mão se as duas versões estiverem lá.
MODO_CLIPE = None

# ── AS CAMADAS DESTE VÍDEO ──────────────────────────────────────────────────
# Declaração ÚNICA: mora no Drive (`<nome>_camadas.json`), junto do vídeo, e
# TODOS os notebooks de legenda a leem e obedecem. A decisão é do vídeo, não de
# quem está rodando -- antes eram quatro interruptores em três notebooks, e
# declarar num e esquecer no outro dava vídeo sem a camada.
#
#   None  = usa o que já está declarado (ou o padrão, na primeira vez)
#   dict  = REDECLARA, e os outros notebooks passam a obedecer a nova
#
# O que cada camada precisa, se estiver ligada:
#   indicador_versiculo  `<nome>_versiculo_multilingue.srt`, da célula de
#                        indicador do caption-multilang-burn.ipynb. Quais
#                        idiomas aparecem sai de config.IDIOMAS_INDICADOR_VERSICULO,
#                        hoje ("en", "ko", "zh") -- um por sistema de escrita
#                        (pt/es/fr abreviam Mateus igual, e repetir não informa).
#   titulo_trecho        `<nome>_titulo.srt`, da última célula do
#                        match-scene-verse.ipynb. Só o inglês vai pra tela; o
#                        coreano e o chinês ficam na planilha, pra descrição.
#   siglas_idioma        nada: sai junto do .ass, lá no generate.
#
# Camada ligada cujo arquivo não existe é ERRO aqui, não aviso: quem ligou
# espera ver a camada no vídeo, e aviso no meio de saída longa passa batido --
# aí o vídeo sai sem ela e só se descobre assistindo.
CAMADAS = None
# CAMADAS = {"siglas_idioma": True, "indicador_versiculo": True, "titulo_trecho": True}

# ── COMO A IMAGEM ENTRA ─────────────────────────────────────────────────────
# "cena_inteira"  a foto/clipe ocupa a tela e a legenda vem por cima. É o de
#                 sempre.
# "miniatura"     a foto sai de trás do texto e vira uma miniatura ABAIXO da
#                 legenda, sobre uma IMAGEM MESTRE limpa que passa a ser o
#                 fundo. Serve pra foto e texto pararem de disputar o mesmo
#                 pixel -- com seis linhas coloridas em cima de uma foto, o
#                 olho não sabe se está lendo ou olhando.
#
# Os dois vídeos convivem na pasta: o de miniatura sai com "_mini" no nome.
MODO_FUNDO = "cena_inteira"
if MODO_FUNDO not in ("cena_inteira", "miniatura"):
    raise ValueError(f"MODO_FUNDO={MODO_FUNDO!r} não existe. Use 'cena_inteira' "
                     f"ou 'miniatura'. (Errar o nome aqui em silêncio faria o "
                     f"vídeo sair no modo de sempre, como se a opção não "
                     f"tivesse sido tocada.)")

# Espaçamento entre as linhas de legenda NA VERSÃO EM MINIATURA. Aperta a
# pilha de idiomas pra sobrar mais tela pra foto -- o .ass é o mesmo, só os
# \pos mudam (nada de rodar o Stanza de novo).
#
# Medido no Mateus 2, com os seis idiomas (cada linha tem ~31px de tinta):
#     80 (o de hoje) folga de 47px entre linhas → miniatura 352x198
#     72             folga de 39px             → miniatura 422x238
#     66 (padrão)    folga de 33px             → miniatura 476x268
#     58             folga de 25px             → miniatura 546x308
# Abaixo de ~25px de folga as faixas coloridas começam a se ler como um bloco
# só, e a cor por idioma perde a serventia. 80 mantém a legenda igual à da
# versão de tela cheia.
ESPACAMENTO_MINIATURA = 66

# A imagem mestre é procurada nesta ordem: a deste vídeo
# (<nome>_imagem_mestre.png, na pasta do vídeo), depois a do canal
# (imagem_mestre.png, em assets/marca). Não achando nenhuma, a queima gera um
# fundo de cor lisa -- dá pra ver a variante hoje e trocar a imagem depois,
# sem mexer em nada.
COR_FUNDO_PROVISORIA = "#F2EEE6"   # papel claro

# Moldura escura em volta da miniatura, em px (0 tira). Numa imagem mestre
# clara, foto sem moldura fica "flutuando"; com 3px ela vira quadro.
BORDA_MINIATURA = 3

# O modo do fundo (imagem parada ou clipe de vídeo) sai do arquivo que EXISTE
# no Drive, não de uma opção que dá pra esquecer de marcar -- mesma ideia do
# sufixo `_zh` lido do nome do .ass. Sem isto, queimar sobre um vídeo base
# feito em modo imagem procurava `_video_base.mp4` e não achava; e se achasse,
# o resultado sairia sem `_img`, por cima da versão de clipe.
import config as _cfgmod
if MODO_CLIPE is None:
    MODO_CLIPE = _cfgmod.detectar_modo_clipe(
        Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/videos/{NOME_ORACAO}"),
        NOME_ORACAO,
    )
    print(f"🎞️  Background mode detected from Drive: {MODO_CLIPE}")

config = PipelineConfig(NOME_ORACAO=NOME_ORACAO, PASTA_DRIVE_RAIZ=PASTA_DRIVE_RAIZ,
                        IDIOMA_MESTRE="en", MODO_CLIPE=MODO_CLIPE)
drive_client = DriveClient.get()

# Nome do vídeo/imagem de fundo (o mesmo que o resto do pipeline gera) —
# vem do config.py, centralizado (config.NOME_VIDEO_BASE)
NOME_VIDEO_BASE = config.NOME_VIDEO_BASE

# ── camadas: uma declaração só, obedecida por todos os notebooks ───────────
import camadas as _cm
_arq_camadas = Path(config.nome_camadas)
drive_client.download(config.pasta_oracao, _arq_camadas.name, _arq_camadas)
CAMADAS_ATIVAS, _mudou = _cm.resolver(_arq_camadas, CAMADAS)
if _mudou:
    _cm.salvar(CAMADAS_ATIVAS, _arq_camadas)
    drive_client.upload(_arq_camadas, config.pasta_oracao, "application/json")
print(_cm.descrever(CAMADAS_ATIVAS, "declarado aqui" if CAMADAS else "do Drive"))

print(f"Vídeo: {NOME_ORACAO}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3. ENVIAR O .ASS (o original do notebook anterior, ou uma       ║
# ║  versão que você corrigiu manualmente — tanto faz, contanto que  ║
# ║  seja um .ass válido)                                            ║
# ╚══════════════════════════════════════════════════════════════════╝
from google.colab import files

print("Selecione o arquivo .ass:")
enviados = files.upload()
nome_ass = list(enviados.keys())[0]
caminho_ass = Path(nome_ass)
print(f"✅ Recebido: {caminho_ass}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4. BAIXAR O VÍDEO/IMAGEM DE FUNDO DO DRIVE                      ║
# ╚══════════════════════════════════════════════════════════════════╝
from srt_utils import ler_srt  # garante que os módulos do pipeline já foram testados no import

video_base_local = Path(NOME_VIDEO_BASE)
ok = drive_client.download(config.pasta_oracao, NOME_VIDEO_BASE, video_base_local)
if not ok:
    raise FileNotFoundError(f"Não achei '{NOME_VIDEO_BASE}' em {config.pasta_oracao} — ajuste NOME_VIDEO_BASE na célula 2")
print(f"✅ Vídeo base baixado: {video_base_local}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5. QUEIMAR                                                      ║
# ╚══════════════════════════════════════════════════════════════════╝
from ffmpeg_utils import queimar_legendas_ass

# O .ass de 6 idiomas sai do caption-multicolor-zh-generate já com `_zh` no
# nome. Sem ler esse sufixo aqui, queimar a versão de 6 idiomas gravaria por
# cima do `_final_multicolor.mp4` de 5 -- mesmo nome, vídeo diferente, e nada
# avisaria: você só descobriria abrindo o arquivo. Por isso a variante do
# resultado vem do nome do arquivo que VOCÊ escolheu, não de uma opção que dá
# pra esquecer de marcar.
config.SUFIXO_VARIANTE_IDIOMAS = "_zh" if caminho_ass.stem.endswith("_zh") else ""
if config.SUFIXO_VARIANTE_IDIOMAS:
    print("🇨🇳 .ass de 6 idiomas detectado — o resultado vai com sufixo _zh")

caminho_saida = Path(config.NOME_VIDEO_FINAL_MULTICOLOR)
if caminho_saida.exists():
    print(f"⚠️  {caminho_saida.name} já existe aqui e vai ser refeito")
print(f"Saída: {caminho_saida.name}")

# ── As camadas dos cantos ──────────────────────────────────────────────────
# queimar_legendas_ass aceita uma LISTA de .ass, encadeados no mesmo filtro:
# um passe de codificação só, então cada camada extra é de graça.
#
# Quem manda aqui é a declaração (CAMADAS_ATIVAS, da célula 2), não um
# interruptor local. E camada ligada SEM o arquivo dela para a queima: antes
# isto imprimia um aviso e queimava assim mesmo, o que é o pior dos dois
# mundos -- você pediu a camada, o vídeo saiu sem ela, e o aviso ficou perdido
# no meio da saída do ffmpeg.
camadas_ass = [caminho_ass]

if CAMADAS_ATIVAS["indicador_versiculo"]:
    from ffmpeg_utils import gerar_ass_versiculo
    _nome_versiculo = config.nome_srt_versiculo_multilingue
    _srt_versiculo = Path(_nome_versiculo)
    drive_client.download(config.pasta_oracao, _nome_versiculo, _srt_versiculo)
    _cm.exigir(CAMADAS_ATIVAS, "indicador_versiculo", _srt_versiculo.exists())
    _legendas_v = ler_srt(_srt_versiculo)
    camadas_ass.append(gerar_ass_versiculo(
        _legendas_v, config,
        caminho_saida=Path(f"versiculo_{NOME_ORACAO}.ass")))
    print(f"📖 Indicador de versículo: {len(_legendas_v)} versículos "
          f"(«{_legendas_v[0].texto}» ... «{_legendas_v[-1].texto}»)")

if CAMADAS_ATIVAS["titulo_trecho"]:
    from ffmpeg_utils import gerar_ass_versiculo as _gerar_ass_canto
    _nome_titulo = config.nome_srt_titulo
    _srt_titulo = Path(_nome_titulo)
    drive_client.download(config.pasta_oracao, _nome_titulo, _srt_titulo)
    _cm.exigir(CAMADAS_ATIVAS, "titulo_trecho", _srt_titulo.exists())
    _legendas_t = ler_srt(_srt_titulo)
    camadas_ass.append(_gerar_ass_canto(
        _legendas_t, config,
        caminho_saida=Path(f"titulo_{NOME_ORACAO}.ass"),
        alinhamento=9,     # canto superior DIREITO (o versículo fica no esquerdo)
        italico=1, negrito=0))
    print(f"📑 Título do trecho: {len(_legendas_t)} faixa(s) "
          f"(«{_legendas_t[0].texto}» ...)")

# ── A imagem vira miniatura? ───────────────────────────────────────────────
# Tudo num passe de codificação só: fundo mestre + miniatura + camadas de
# legenda. Compor num passe e queimar noutro custaria uma recodificação
# inteira e mais uma geração de perda.
_mestre, _caixa = None, None
if MODO_FUNDO == "miniatura":
    import moldura as _mo

    # A legenda desce de posição ANTES de a caixa ser calculada -- a caixa sai
    # do .ass que vai ser queimado, então ela já nasce sabendo quantos idiomas
    # tem e com que espaçamento.
    if ESPACAMENTO_MINIATURA != 80:
        _ass_mini = Path(f"{caminho_ass.stem}_mini.ass")
        camadas_ass[0] = _mo.reposicionar_ass(
            camadas_ass[0], _ass_mini, _mo.mapa_de_reposicionamento(ESPACAMENTO_MINIATURA))
        print(f"📐 Legenda reposicionada: espaçamento {ESPACAMENTO_MINIATURA}px "
              f"(era 80) → {_ass_mini.name}")
    _caixa = _mo.caixa_para_ass(camadas_ass[0])

    # a deste vídeo primeiro, depois a do canal, depois cor lisa
    _mestre = Path(config.nome_imagem_mestre)
    if drive_client.download(config.pasta_oracao, _mestre.name, _mestre) and _mestre.exists():
        print(f"🖼️  Imagem mestre deste vídeo: {_mestre.name}")
    else:
        _mestre = Path(config.NOME_IMAGEM_MESTRE_PADRAO)
        if drive_client.download(config.pasta_assets_logo, _mestre.name, _mestre) \
                and _mestre.exists():
            print(f"🖼️  Imagem mestre do canal: {_mestre.name}")
        else:
            _mestre = _mo.gerar_fundo_liso(Path("fundo_provisorio.png"), COR_FUNDO_PROVISORIA)
            print(f"🎨 Sem imagem mestre no Drive — fundo liso {COR_FUNDO_PROVISORIA}.")
            print(f"    Pra trocar: suba a imagem como '{config.nome_imagem_mestre}'")
            print(f"    em {config.pasta_oracao}, ou como "
                  f"'{config.NOME_IMAGEM_MESTRE_PADRAO}' em {config.pasta_assets_logo}.")

    config.SUFIXO_VARIANTE_LAYOUT = "_mini"
    caminho_saida = Path(config.NOME_VIDEO_FINAL_MULTICOLOR)
    print(f"🖼️  Miniatura {_caixa} — saída: {caminho_saida.name}")

resultado = queimar_legendas_ass(
    video_entrada=video_base_local,
    ass_path=camadas_ass,
    saida=caminho_saida,
    imagem_mestre=_mestre,
    miniatura=tuple(_caixa) if _caixa else None,
    borda_miniatura=BORDA_MINIATURA,
)
print(f"✅ Vídeo com legenda colorida: {resultado}")
print("\nRode a célula 6 pra baixar e conferir. Só rode a célula 7 (salvar no Drive) depois de confirmar que ficou bom.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  6. BAIXAR O RESULTADO (pra conferir)                            ║
# ╚══════════════════════════════════════════════════════════════════╝
files.download(str(resultado))


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  7. SALVAR NO DRIVE — só rode depois de conferir que ficou bom   ║
# ╚══════════════════════════════════════════════════════════════════╝
caminho_no_drive = drive_client.upload(resultado, config.pasta_oracao)
print(f"✅ Salvo no Drive: {caminho_no_drive}")
